# Saudi Arabic STT Fine-Tuning
## Fine-tune MasriSwitch-Gemma3n for Saudi Dialect + Noise Robustness

**Base model**: `oddadmix/MasriSwitch-Gemma3n-Transcriber-v1` (Gemma3n-E4B, 8B params)
**Dataset**: SADA (MohamedRashad/SADA22) — 667h Saudi Arabic speech
**Requirements**: GPU with 16GB+ VRAM (A100 recommended)

### Pipeline
1. Install dependencies
2. Load & prepare SADA Saudi dialect data (correct field values: `najidi`, `hijazi`, `khaliji`)
3. Data analysis
4. Noise augmentation (research-backed: SNR 5-15 dB, speed perturbation 0.9/1.0/1.1)
5. Load model + LoRA (with audio-specific modules)
6. Format data + **proper label masking** (`train_on_responses_only`)
7. Phase 1: Saudi dialect adaptation (LR 2e-5, 2 epochs, clean data)
8. Phase 2: Noise robustness (LR 1e-5, 1 epoch, augmented data)
9. Evaluation (WER, CER, MER, WIL, WIP, RTF)
10. Merge & export

### Key fixes vs. basic notebooks
- **Label masking**: Uses `train_on_responses_only` — only trains on assistant transcription, NOT the user prompt
- **SADA dialect strings**: Uses actual field values (`najidi`, not `Najdi`)
- **SNR range**: 5-15 dB (NVIDIA NeMo / SADA paper standard), not 3-40 dB
- **SFTConfig**: Includes `dataset_text_field=""` and `dataset_kwargs={"skip_prepare_dataset": True}`

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install transformers datasets torch trl accelerate bitsandbytes peft
!pip install librosa soundfile audiomentations
!pip install jiwer>=3.0 whisper-normalizer

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load & Prepare SADA Saudi Dialect Data

In [ ]:
from datasets import load_dataset
import re
import unicodedata
import numpy as np
from collections import Counter

# ── Arabic text normalization ─────────────────────────────────
# Conservative normalization for training (preserve dialect info)
def normalize_arabic_for_training(text):
    """Normalize Arabic text for training labels (conservative)."""
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'[\u064B-\u065F\u0670\u0617-\u061A]', '', text)  # Remove diacritics
    text = re.sub(r'[\u0623\u0625\u0622\u0671]', '\u0627', text)    # Normalize alef variants
    text = re.sub(r'\u0649', '\u064A', text)                         # Alef maqsura → yaa
    text = re.sub(r'\u0640', '', text)                                # Remove tatweel
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Aggressive normalization for evaluation (OALL standard)
def normalize_arabic_for_eval(text):
    """Normalize Arabic text for WER/CER evaluation."""
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'[\u064B-\u065F\u0670\u0617-\u061A]', '', text)
    text = re.sub(r'[\u0623\u0625\u0622\u0671]', '\u0627', text)
    text = re.sub(r'\u0649', '\u064A', text)
    text = re.sub(r'\u0640', '', text)
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077Fa-zA-Z0-9\s]', ' ', text)  # Remove punctuation
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Load SADA dataset ────────────────────────────────────────
print("Loading SADA dataset (first time downloads ~50GB)...")
sada = load_dataset("MohamedRashad/SADA22", split="train")
print(f"Total samples: {len(sada)}")

# Check dialect distribution
dialects = Counter(sada["speaker_dialect"])
for d, c in dialects.most_common():
    print(f"  {d}: {c}")

In [ ]:
# ── Filter for Saudi dialects ─────────────────────────────────
# CORRECT field values in SADA22 (verified from dataset):
#   "najidi", "hijazi", "khaliji" (lowercase, "najidi" not "najdi")
saudi_dialect_keywords = ["najidi", "hijazi", "khaliji"]

def is_saudi(example):
    dialect = str(example.get("speaker_dialect", "")).lower()
    return any(kw in dialect for kw in saudi_dialect_keywords)

saudi_data = sada.filter(is_saudi)
print(f"Saudi dialect samples: {len(saudi_data)}")

# If too few Saudi-specific samples, use all SADA (it's all from Saudi Broadcasting Authority)
if len(saudi_data) < 5000:
    print(f"WARNING: Only {len(saudi_data)} Saudi-specific samples found.")
    print("Using all SADA data (all sourced from Saudi broadcasting).")
    saudi_data = sada
    print(f"Using all SADA: {len(saudi_data)} samples")

In [ ]:
# ── Filter by quality ─────────────────────────────────────────
def good_sample(example):
    # Duration: 2-30 seconds (SADA paper: <2s → >100% CER; Gemma3n handles up to 30s)
    audio = example["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    if not (2.0 <= duration <= 30.0):
        return False
    # Text: at least 2 Arabic characters
    text = example.get("cleaned_text", example.get("text", ""))
    arabic_chars = re.findall(r'[\u0600-\u06FF]', text)
    return len(arabic_chars) >= 2

saudi_data = saudi_data.filter(good_sample)
print(f"After quality filter: {len(saudi_data)}")

# ── Prepare transcripts ──────────────────────────────────────
# Use cleaned_text field (SADA provides both text and cleaned_text)
saudi_data = saudi_data.map(lambda x: {
    "transcript": normalize_arabic_for_training(x.get("cleaned_text", x["text"])),
    "duration_seconds": round(len(x["audio"]["array"]) / x["audio"]["sampling_rate"], 2),
})

total_hours = sum(saudi_data["duration_seconds"]) / 3600
avg_duration = sum(saudi_data["duration_seconds"]) / len(saudi_data)
print(f"Total duration: {total_hours:.1f} hours")
print(f"Average sample: {avg_duration:.1f} seconds")

In [ ]:
# ── Split train/eval ──────────────────────────────────────────
split = saudi_data.train_test_split(
    test_size=min(1000, int(len(saudi_data) * 0.05)),
    seed=42,
)
train_data = split["train"]
eval_data = split["test"]
print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

# ── Data analysis ─────────────────────────────────────────────
durations = train_data["duration_seconds"]
print(f"\nDuration stats:")
print(f"  Min: {min(durations):.1f}s  Max: {max(durations):.1f}s")
print(f"  Mean: {np.mean(durations):.1f}s  Median: {np.median(durations):.1f}s")

# Show a few transcripts
print("\nSample transcripts:")
for i in range(min(5, len(train_data))):
    print(f"  [{train_data[i]['duration_seconds']:.1f}s] {train_data[i]['transcript'][:80]}")

## 3. Noise Augmentation (Research-Backed)

**Research basis:**
- SNR range: **5-15 dB** (NVIDIA NeMo standard, SADA ASR paper)
- Speed perturbation: **0.9, 1.0, 1.1** (Povey 2015, industry standard)
- Target ratio: **~50:50 clean:noisy** (multi-condition training standard)
- SADA paper finding: training on noisy data directly > denoising first

In [ ]:
from audiomentations import Compose, AddGaussianSNR, TimeStretch, Gain, BandPassFilter
from datasets import Dataset, concatenate_datasets
from tqdm.notebook import tqdm

# ── Research-backed augmentation pipeline ─────────────────────
# SNR 5-15 dB: noisy enough for robustness, clean enough for valid transcripts
noise_augment = Compose([
    AddGaussianSNR(min_snr_db=5.0, max_snr_db=15.0, p=0.7),
    Gain(min_gain_db=-4, max_gain_db=4, p=0.3),
    BandPassFilter(min_center_freq=200, max_center_freq=3500, p=0.15),
])

# Speed perturbation factors (Povey 2015)
SPEED_FACTORS = [0.9, 1.0, 1.1]

# Create augmented copies (target: ~50:50 clean:noisy ratio)
augmented = []
for example in tqdm(train_data, desc="Augmenting"):
    audio = np.array(example["audio"]["array"], dtype=np.float32)
    sr = example["audio"]["sampling_rate"]

    # Speed perturbation (30% chance)
    if np.random.random() < 0.3:
        factor = np.random.choice(SPEED_FACTORS)
        if factor != 1.0:
            ts = TimeStretch(min_rate=factor, max_rate=factor, p=1.0)
            audio = ts(samples=audio, sample_rate=sr)

    # Add noise
    aug_audio = noise_augment(samples=audio, sample_rate=sr)
    aug_audio = np.clip(aug_audio, -1.0, 1.0)

    augmented.append({
        "audio": {"array": aug_audio, "sampling_rate": sr},
        "transcript": example["transcript"],
        "duration_seconds": example["duration_seconds"],
    })

print(f"Original (clean): {len(train_data)} | Augmented (noisy): {len(augmented)}")

# Combine clean + noisy for Phase 2 training
aug_dataset = Dataset.from_list(augmented)
cols = ["audio", "transcript", "duration_seconds"]
train_clean = train_data.select_columns([c for c in cols if c in train_data.column_names])
train_noisy = concatenate_datasets([train_clean, aug_dataset]).shuffle(seed=42)
print(f"Phase 2 combined: {len(train_noisy)} samples (~50:50 clean:noisy)")

## 4. Load Model & Configure LoRA

**LoRA target modules** include audio-specific layers (`post`, `linear_start`, `linear_end`, `embedding_projection`) — CRITICAL for STT fine-tuning.

**modules_to_save** includes `embed_audio` — required to properly adapt audio embeddings.

In [ ]:
import torch
torch._dynamo.config.cache_size_limit = 32

from unsloth import FastModel

model, processor = FastModel.from_pretrained(
    model_name="oddadmix/MasriSwitch-Gemma3n-Transcriber-v1",
    dtype=None,
    max_seq_length=1024,
    load_in_4bit=True,
    full_finetuning=False,
)

model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        # Attention
        "q_proj", "k_proj", "v_proj", "o_proj",
        # FFN
        "gate_proj", "up_proj", "down_proj",
        # Audio-specific (CRITICAL for STT fine-tuning)
        "post", "linear_start", "linear_end",
        "embedding_projection",
    ],
    modules_to_save=[
        "lm_head",
        "embed_tokens",
        "embed_audio",  # CRITICAL: must save audio embeddings
    ],
    bias="none",
)

model.print_trainable_parameters()

## 5. Format Data & Create Trainer

**CRITICAL FIX**: Uses `train_on_responses_only` to mask everything except the assistant's transcription response. Without this, the model wastes capacity learning to predict the user prompt and system message.

**SFTConfig requirements**:
- `dataset_text_field=""` — required when using custom collate
- `dataset_kwargs={"skip_prepare_dataset": True}` — prevents SFTTrainer from re-processing data

In [ ]:
# ── Format data into Gemma3n chat template ───────────────────
def format_for_training(example):
    messages = [
        {"role": "system", "content": [
            {"type": "text", "text": "You are an assistant that transcribes speech accurately."}
        ]},
        {"role": "user", "content": [
            {"type": "audio", "audio": example["audio"]["array"]},
            {"type": "text", "text": "Please transcribe this audio."}
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": example["transcript"]}
        ]}
    ]
    return {"messages": messages}

# Format both clean (Phase 1) and noisy (Phase 2) datasets
train_p1 = train_data.map(format_for_training, remove_columns=train_data.column_names)
train_p2 = train_noisy.map(format_for_training, remove_columns=train_noisy.column_names)
eval_formatted = eval_data.map(format_for_training, remove_columns=eval_data.column_names)

print(f"Phase 1 train: {len(train_p1)} | Phase 2 train: {len(train_p2)} | Eval: {len(eval_formatted)}")

# ── Collate function ──────────────────────────────────────────
def collate_fn(examples):
    texts, audios = [], []
    for ex in examples:
        texts.append(processor.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        ).strip())
        for msg in ex["messages"]:
            if msg["role"] == "user":
                for c in msg["content"]:
                    if c.get("type") == "audio":
                        audios.append(c["audio"])

    batch = processor(text=texts, audio=audios, return_tensors="pt", padding=True)

    # Create labels (mask padding and audio tokens)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    if hasattr(processor.tokenizer, "audio_token_id") and processor.tokenizer.audio_token_id is not None:
        labels[labels == processor.tokenizer.audio_token_id] = -100
    batch["labels"] = labels
    return batch

In [ ]:
from trl import SFTTrainer, SFTConfig

# ── Phase 1: Saudi Dialect Adaptation (clean data) ────────────
training_args_p1 = SFTConfig(
    output_dir="./saudi_stt_phase1",

    # Batch
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,  # Effective batch size = 4

    # Learning rate
    learning_rate=2e-5,             # Conservative for dialect adaptation
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Duration
    num_train_epochs=2,

    # Memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    optim="adamw_8bit",

    # Sequence
    max_seq_length=1024,

    # Logging
    logging_steps=10,
    logging_first_step=True,
    report_to="none",

    # Saving
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,

    # Evaluation
    eval_strategy="steps",
    eval_steps=500,

    # Misc
    weight_decay=0.01,
    remove_unused_columns=False,
    dataloader_num_workers=4,
    seed=42,

    # REQUIRED for custom collate with SFTTrainer
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer_p1 = SFTTrainer(
    model=model,
    args=training_args_p1,
    train_dataset=train_p1,
    eval_dataset=eval_formatted,
    data_collator=collate_fn,
)

# ── CRITICAL: mask labels to only train on assistant response ──
# Without this, the model trains on system message + user prompt,
# wasting capacity and hurting transcription quality.
from unsloth.chat_templates import train_on_responses_only
trainer_p1 = train_on_responses_only(
    trainer_p1,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

# NOTE: Initial loss of 6-7 is NORMAL for multimodal models (Unsloth docs).
# It should decrease to 3-4 during training.
print("Starting Phase 1: Saudi Dialect Adaptation")
print(f"  Train: {len(train_p1)} clean samples | Eval: {len(eval_formatted)}")
print(f"  LR: 2e-5 | Epochs: 2 | Effective batch: 4")
trainer_p1.train()

## 6. Phase 2: Noise Robustness Training

Lower learning rate (1e-5 = half of Phase 1) to preserve Saudi dialect knowledge while learning noise robustness.

In [ ]:
# Save Phase 1 checkpoint
trainer_p1.save_model("./saudi_stt_phase1/final")
processor.save_pretrained("./saudi_stt_phase1/final")
print("Phase 1 saved.")

# ── Phase 2: Noise Robustness (augmented data) ───────────────
training_args_p2 = SFTConfig(
    output_dir="./saudi_stt_phase2",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=1e-5,             # Half of Phase 1 (preserve knowledge)
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    num_train_epochs=1,             # 1 epoch on augmented data

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    optim="adamw_8bit",

    max_seq_length=1024,
    logging_steps=10,
    logging_first_step=True,
    report_to="none",

    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,

    eval_strategy="steps",
    eval_steps=500,

    weight_decay=0.01,
    remove_unused_columns=False,
    dataloader_num_workers=4,
    seed=42,

    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer_p2 = SFTTrainer(
    model=model,
    args=training_args_p2,
    train_dataset=train_p2,
    eval_dataset=eval_formatted,
    data_collator=collate_fn,
)

# Apply label masking for Phase 2 as well
trainer_p2 = train_on_responses_only(
    trainer_p2,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print("Starting Phase 2: Noise Robustness")
print(f"  Train: {len(train_p2)} samples (clean+noisy) | Eval: {len(eval_formatted)}")
print(f"  LR: 1e-5 | Epochs: 1 | Effective batch: 4")
trainer_p2.train()

# Save Phase 2
trainer_p2.save_model("./saudi_stt_phase2/final")
processor.save_pretrained("./saudi_stt_phase2/final")
print("Phase 2 saved.")

## 7. Evaluation (Full Metrics)

Evaluates with all standard ASR metrics:
- **WER** (Word Error Rate) — primary metric
- **CER** (Character Error Rate)
- **MER** (Match Error Rate, bounded [0,1])
- **WIL** (Word Information Lost)
- **WIP** (Word Information Preserved)
- **RTF** (Real-Time Factor)

Uses OALL-standard Arabic normalization for fair comparison.

In [ ]:
import time
from jiwer import process_words, process_characters

model.eval()

# Use Google's processor for inference (known bug: Unsloth's saved version may fail)
from transformers import AutoProcessor
eval_processor = AutoProcessor.from_pretrained("google/gemma-3n-E4B-it")

# ── Transcribe eval set ───────────────────────────────────────
predictions, references = [], []
total_audio_sec, total_inference_sec = 0.0, 0.0

n_eval = min(200, len(eval_data))
for i, example in enumerate(tqdm(eval_data.select(range(n_eval)), desc="Evaluating")):
    audio = np.array(example["audio"]["array"], dtype=np.float32)
    sr = example["audio"]["sampling_rate"]
    duration = len(audio) / sr

    msgs = [
        {"role": "system", "content": [
            {"type": "text", "text": "You are an assistant that transcribes speech accurately."}
        ]},
        {"role": "user", "content": [
            {"type": "audio", "audio": audio},
            {"type": "text", "text": "Please transcribe this audio."}
        ]},
    ]

    inputs = eval_processor.apply_chat_template(
        msgs, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=256, do_sample=False)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inf_time = time.perf_counter() - t0

    pred = eval_processor.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # Normalize with OALL standard
    ref_norm = normalize_arabic_for_eval(example["transcript"])
    pred_norm = normalize_arabic_for_eval(pred)

    references.append(ref_norm)
    predictions.append(pred_norm)
    total_audio_sec += duration
    total_inference_sec += inf_time

    if i < 5:
        print(f"\n--- Sample {i} ---")
        print(f"  REF: {example['transcript']}")
        print(f"  PRD: {pred}")

# ── Compute all metrics ──────────────────────────────────────
valid = [(r, h) for r, h in zip(references, predictions) if r.strip()]
refs, hyps = zip(*valid) if valid else ([], [])
refs, hyps = list(refs), list(hyps)

if refs:
    w = process_words(refs, hyps)
    c = process_characters(refs, hyps)

    S, D, I, C = w.substitutions, w.deletions, w.insertions, w.hits
    N = S + D + C
    P = S + I + C

    cS, cD, cI, cC = c.substitutions, c.deletions, c.insertions, c.hits
    cN = cS + cD + cC

    wer_val = (S + D + I) / N if N > 0 else 0.0
    cer_val = (cS + cD + cI) / cN if cN > 0 else 0.0
    mer_val = (S + D + I) / (N + I) if (N + I) > 0 else 0.0
    wil_val = 1.0 - (C / N) * (C / P) if N > 0 and P > 0 else 1.0
    wip_val = 1.0 - wil_val
    rtf = total_inference_sec / total_audio_sec if total_audio_sec > 0 else 0

    print("\n" + "=" * 60)
    print("               EVALUATION RESULTS")
    print("=" * 60)
    print(f"  WER  (Word Error Rate):            {wer_val*100:.2f}%")
    print(f"  CER  (Character Error Rate):       {cer_val*100:.2f}%")
    print(f"  MER  (Match Error Rate):           {mer_val*100:.2f}%")
    print(f"  WIL  (Word Information Lost):      {wil_val*100:.2f}%")
    print(f"  WIP  (Word Information Preserved): {wip_val*100:.2f}%")
    print(f"  RTF  (Real-Time Factor):           {rtf:.4f}  ({1/rtf:.1f}x real-time)")
    print(f"\n  Samples: {len(valid)} valid, {len(references)-len(valid)} skipped")
    print(f"  Word alignment: C={C}  S={S}  D={D}  I={I}")
    print("=" * 60)
else:
    print("No valid samples for evaluation.")

## 8. Merge & Export

Merge LoRA adapters into the base model for production deployment.

In [ ]:
# Merge LoRA adapters into base model for production
model.save_pretrained_merged(
    "./saudi_stt_merged",
    processor,
    save_method="merged_16bit",
)
print("Model merged and saved to ./saudi_stt_merged")
print("\nFor inference, use Google's processor (not Unsloth's):")
print("  processor = AutoProcessor.from_pretrained('google/gemma-3n-E4B-it')")

# Optional: Push to HuggingFace Hub
# model.push_to_hub_merged(
#     "your-username/saudi-stt-gemma3n",
#     processor,
#     save_method="merged_16bit",
#     token="your-hf-token",
# )